# Notebook 07 — Machine Learning Preparation

## Objective

This notebook prepares the analytical dataset for predictive customer churn modelling.

The objective is to construct a clean and reproducible preprocessing pipeline before training any machine-learning algorithm.

The preparation includes feature selection, train-test split, preprocessing pipelines, categorical encoding and numerical imputation while avoiding any information leakage.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "dataset_modelisation.parquet"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "models"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
df = pd.read_parquet(DATA_FILE)

print("=" * 70)
print("MACHINE LEARNING PREPARATION")
print("=" * 70)

print(df.shape)

display(df.head())

MACHINE LEARNING PREPARATION
(195120, 20)


,CUSTOMER_NO,CHURN,AGE,TRANCHE_AGE,ANCIENNETE_CLIENT_ANNEES,SALAIRE,HAS_SALAIRE,EST_TUNISIEN,EST_RESIDENT,SITUATION_FAMILIALE,TYPE_CLIENT,SEGMENT,DOSSIER_COMPLET,SCORE_KYC,KYC_RISQUE_ELEVE,JOURS_DEPUIS_REVUE,JOURS_AVANT_PROCHAINE_REVUE,REVUE_EN_RETARD,A_HISTORIQUE_REVUE,NB_COMPTES
0,113425593,0,67.2,65+,13.1,NaN,False,True,True,M,PPH,Retail,False,LR,False,2098.0,-637.0,True,True,1
1,113198502,0,80.7,65+,7.0,NaN,False,True,True,C,PPH,Retail,True,LR,False,32.0,1428.0,False,True,2
2,113377287,0,46.3,36-50,11.5,400.0,True,True,True,C,PPH,Retail,True,LR,False,106.0,1354.0,False,True,1
3,113392283,0,34.3,26-35,12.0,400.0,True,True,True,C,PPH,Retail,False,MR,False,657.0,439.0,False,True,1
4,113325457,0,15.9,15-25,8.3,NaN,False,True,True,C,PPH,Retail,False,MR,False,1680.0,-585.0,True,True,1


In [4]:
TARGET = "CHURN"

ID_COLUMN = "CUSTOMER_NO"

In [5]:
X = df.drop(
    columns=[
        TARGET,
        ID_COLUMN,
        "JOURS_DEPUIS_REVUE"
    ]
)
    
y = df[TARGET]

print(X.shape)
print(y.shape)

(195120, 17)
(195120,)


In [6]:
categorical_features = (
    X.select_dtypes(
        include=[
            "object",
            "category",
            "bool"
        ]
    )
    .columns
    .tolist()
)

numeric_features = (
    X.select_dtypes(
        include=[
            "number"
        ]
    )
    .columns
    .tolist()
)

C:\Users\Sarra\AppData\Local\Temp\ipykernel_16632\2884473017.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X.select_dtypes(


In [7]:
print(categorical_features)

print()

print(numeric_features)


['TRANCHE_AGE', 'HAS_SALAIRE', 'EST_TUNISIEN', 'EST_RESIDENT', 'SITUATION_FAMILIALE', 'TYPE_CLIENT', 'SEGMENT', 'DOSSIER_COMPLET', 'SCORE_KYC', 'KYC_RISQUE_ELEVE', 'REVUE_EN_RETARD', 'A_HISTORIQUE_REVUE']

['AGE', 'ANCIENNETE_CLIENT_ANNEES', 'SALAIRE', 'JOURS_AVANT_PROCHAINE_REVUE', 'NB_COMPTES']


In [8]:
X_train,\
X_test,\
y_train,\
y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    stratify=y,

    random_state=42

)

In [9]:
print(len(X_train))

print(len(X_test))

156096
39024


In [10]:
# ==========================================================
# NUMERICAL PREPROCESSING
# ==========================================================

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

In [11]:
# ==========================================================
# CATEGORICAL PREPROCESSING
# ==========================================================

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [12]:
# ==========================================================
# COMPLETE PREPROCESSING PIPELINE
# ==========================================================

preprocessor = ColumnTransformer(

    transformers=[

        (
            "num",
            numeric_transformer,
            numeric_features
        ),

        (
            "cat",
            categorical_transformer,
            categorical_features
        )

    ]

)

In [13]:
print("=" * 70)
print("PREPROCESSING SUMMARY")
print("=" * 70)

print(f"Numerical features   : {len(numeric_features)}")
print(f"Categorical features : {len(categorical_features)}")

print("\nNumerical")

print(numeric_features)

print("\nCategorical")

print(categorical_features)

PREPROCESSING SUMMARY
Numerical features   : 5
Categorical features : 12

Numerical
['AGE', 'ANCIENNETE_CLIENT_ANNEES', 'SALAIRE', 'JOURS_AVANT_PROCHAINE_REVUE', 'NB_COMPTES']

Categorical
['TRANCHE_AGE', 'HAS_SALAIRE', 'EST_TUNISIEN', 'EST_RESIDENT', 'SITUATION_FAMILIALE', 'TYPE_CLIENT', 'SEGMENT', 'DOSSIER_COMPLET', 'SCORE_KYC', 'KYC_RISQUE_ELEVE', 'REVUE_EN_RETARD', 'A_HISTORIQUE_REVUE']


In [14]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

print(X_train_processed.shape)
print(X_test_processed.shape)

(156096, 41)
(39024, 41)


In [15]:
# ==========================================================
# FINAL FEATURE INVENTORY
# ==========================================================

feature_summary = pd.DataFrame({
    "FEATURE": X.columns,
    "TYPE": [
        "Numerical" if c in numeric_features else "Categorical"
        for c in X.columns
    ]
})

display(feature_summary)

feature_summary.to_excel(
    PROJECT_ROOT / "outputs" / "tables" / "ml_features.xlsx",
    index=False
)

,FEATURE,TYPE
0,AGE,Numerical
1,TRANCHE_AGE,Categorical
2,ANCIENNETE_CLIENT_ANNEES,Numerical
3,SALAIRE,Numerical
4,HAS_SALAIRE,Categorical
5,EST_TUNISIEN,Categorical
6,EST_RESIDENT,Categorical
7,SITUATION_FAMILIALE,Categorical
8,TYPE_CLIENT,Categorical
9,SEGMENT,Categorical


In [16]:
from pathlib import Path
import joblib

PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

pipeline_bundle = {
    "pipeline": preprocessor,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
}

OUTPUT_FILE = MODEL_DIR / "preprocessing_pipeline.joblib"

joblib.dump(
    pipeline_bundle,
    OUTPUT_FILE
)

print(f"Preprocessing pipeline saved successfully:")
print(OUTPUT_FILE)
print(f"File exists: {OUTPUT_FILE.exists()}")

Preprocessing pipeline saved successfully:
c:\Users\Sarra\OneDrive\Desktop\PFE_CHURN_ESB\models\preprocessing_pipeline.joblib
File exists: True


# Machine Learning Preparation Summary

The analytical dataset has been successfully prepared for predictive modelling.

Main preprocessing steps:

- Removal of identifier columns.
- Separation of features and target.
- Stratified train-test split.
- Median imputation for numerical variables.
- Most frequent imputation for categorical variables.
- One-Hot Encoding of categorical features.
- Construction and persistence of a reusable preprocessing pipeline.

The dataset is now ready for supervised machine-learning algorithms.